# 1-bob. 1-BOB. SUN'IY INTELLEKT VA KIBERXAVFSIZLIKKA KIRISH
## Kengaytirilgan amaliy ish: chegarani ogohlantirishlar byudjeti asosida tanlash

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/01-bob/1-2_chegarani-ogohlantirishlar-byudjeti-asosida-tanlash.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 1-bob.*

> Bu kengaytirilgan amaliy ish darslikning **60610700 – Sun'iy intellekt yo'nalishi versiyasida** keltirilgan.

Yuqoridagi misolda model atigi sakkizta xabarda o'qitilgan edi. Endi vazifani real sharoitga yaqinlashtiramiz. Dastur shablonlar asosida 3300 ta o'quv va 10 200 ta sinov xabarini yaratadi. Sinov to'plamida o'qitishda uchramagan yangi firibgarlik kampaniyasi (soxta jarima, posilka va soliq qaytarish xabarlari) hamda firibgarlikka o'xshash, lekin oddiy xabarlar (bank bildirishnomalari, aksiyalar) bor. Matn belgilar n-gramlari va logistik regressiya yordamida tasniflanadi, so'ngra uch xil yondashuv taqqoslanadi: qoida, standart 0,5 chegarali model va chegarasi ogohlantirishlar byudjetiga qarab tanlangan model.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

rng = np.random.default_rng(1)

In [2]:
# 1. Sun'iy SMS to'plami: shablonlar + tasodifiy o'zgarishlar
firib = ["Tabriklaymiz! Siz {s} yutdingiz, {k}", "Kartangiz bloklandi. {k}",
         "Bepul {s}! Havolaga kiring: {h}", "Hisobingizga {s} tushdi, tasdiqlash uchun {k}",
         "Bank xavfsizlik xizmati: shubhali to'lov, {k}"]
yangi_firib = ["Davlat xizmatlari: jarima to'lanmagan, to'lash uchun {h2}",
               "Sizga posilka keldi, manzilni tasdiqlang: {h2}",
               "Soliq qaytarildi, olish uchun ma'lumotlarni kiriting: {h2}"]   # o'qitishda bo'lmagan kampaniya       # o'qitishda bo'lmagan kampaniya
oddiy = ["Ertaga soat {t} da darsga kelasanmi?", "Onam {n} olib kelishimni so'radi",
         "Kutubxonadan {n} olib qo'ydim", "Bugun amaliy mashg'ulot {x}-xonada bo'ladi",
         "To'lov qabul qilindi. Rahmat, {n} buyurtmangiz yo'lda",
         "Kodni hech kimga aytmang: {c}. Bank xodimlari kod so'ramaydi",
         "Kartangizga {s} tushdi. Balans: {c} so'm",              # firibgarlikka o'xshash, lekin oddiy
         "Xaridingiz uchun {s}! Aksiya 31-dekabrgacha",
         "Posilkangiz {x}-pochta bo'limiga keldi", "Soliq hisoboti qabul qilindi, rahmat"]
s = ["10 mln so'm", "iPhone", "sovg'a", "bonus", "500 ming so'm"]
k = ["SMS kodni ayting", "kodni yuboring", "karta raqamini yuboring", "operatorga kodni ayting"]
h = ["bit.ly/x7", "uzcard-bonus.top", "my-gov-uz.site", "posilka-uz.click"]
n = ["non", "kitob", "daftar", "dori"]
h2 = ["e-jarima-uz.info", "pochta-uz.delivery", "soliq-qaytarish.online"]

def yasash(shablonlar, soni):
    out = []
    for _ in range(soni):
        t = rng.choice(shablonlar)
        out.append(t.format(s=rng.choice(s), k=rng.choice(k), h=rng.choice(h), h2=rng.choice(h2), n=rng.choice(n),
                            t=rng.integers(8, 18), x=rng.integers(100, 400), c=rng.integers(1000, 9999)))
    return out

X_o = yasash(firib, 300) + yasash(oddiy, 3000)            # o'qitish: 300 firibgar, 3000 oddiy
y_o = np.r_[np.ones(300), np.zeros(3000)]
X_s = yasash(firib, 100) + yasash(yangi_firib, 100) + yasash(oddiy, 10000)
y_s = np.r_[np.ones(200), np.zeros(10000)]
yangi = np.r_[np.zeros(100), np.ones(100), np.zeros(10000)].astype(bool)  # yangi kampaniya xabarlari

In [3]:
# 2. Qoida va model
def qoida(m):
    m = m.lower()
    return ("yutdingiz" in m) or ("kodni ayting" in m) or ("kodni yuboring" in m)

vek = TfidfVectorizer(analyzer="char_wb", ngram_range=(3, 5))
model = LogisticRegression(max_iter=1000, class_weight="balanced").fit(vek.fit_transform(X_o), y_o)
ehtimol = model.predict_proba(vek.transform(X_s))[:, 1]
q = np.array([qoida(m) for m in X_s])

In [4]:
# 3. Ogohlantirishlar byudjeti: tahlilchilar kuniga ko'rib chiqa oladigan ogohlantirishlar soni
def chegara(byudjet):
    return np.sort(ehtimol)[::-1][byudjet - 1]
def hisobot(nom, ogoh):
    print(f"{nom:<28} ogohlantirish={ogoh.sum():>5}  aniqlangan={int((ogoh & (y_s == 1)).sum()):>3}/200"
          f"  shundan yangi kampaniya={int((ogoh & yangi).sum()):>3}/100"
          f"  noto'g'ri={int((ogoh & (y_s == 0)).sum()):>4}")
hisobot("Qoida", q)
hisobot("Model, chegara 0,5", ehtimol >= 0.5)
for b in (250, 1000):
    print(f"   byudjet {b}: chegara = {chegara(b):.3f}")
    hisobot(f"Model, byudjet {b}", ehtimol >= chegara(b))

Qoida                        ogohlantirish=   70  aniqlangan= 70/200  shundan yangi kampaniya=  0/100  noto'g'ri=   0
Model, chegara 0,5           ogohlantirish=  120  aniqlangan=120/200  shundan yangi kampaniya= 20/100  noto'g'ri=   0
   byudjet 250: chegara = 0.046
Model, byudjet 250           ogohlantirish=  250  aniqlangan=200/200  shundan yangi kampaniya=100/100  noto'g'ri=  50
   byudjet 1000: chegara = 0.032
Model, byudjet 1000          ogohlantirish= 1000  aniqlangan=200/200  shundan yangi kampaniya=100/100  noto'g'ri= 800


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Qoida                        ogohlantirish=   70  aniqlangan= 70/200  shundan yangi kampaniya=  0/100  noto'g'ri=   0
Model, chegara 0,5           ogohlantirish=  120  aniqlangan=120/200  shundan yangi kampaniya= 20/100  noto'g'ri=   0
   byudjet 250: chegara = 0.046
Model, byudjet 250           ogohlantirish=  250  aniqlangan=200/200  shundan yangi kampaniya=100/100  noto'g'ri=  50
   byudjet 1000: chegara = 0.032
Model, byudjet 1000          ogohlantirish= 1000  aniqlangan=200/200  shundan yangi kampaniya=100/100  noto'g'ri= 800
```

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 1-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.